<img src="https://www.epfl.ch/about/overview/wp-content/uploads/2020/07/logo-epfl-1024x576.png" width="140px" alt="EPFL_logo">

## Image Processing Laboratory Notebooks
---

This Jupyter notebook is part of a series of computer laboratories which are designed
to teach image-processing programming; they are running on the EPFL's Noto server. They are the practical complement of the theoretical lectures of the EPFL's Master course 
[**MICRO-511 Image Processing I**](https://moodle.epfl.ch/course/view.php?id=522) taught by Prof. M. Unser and Prof. D. Van de Ville.

The project is funded by the Center for Digital Education and the School of Engineering. It is owned by the [Biomedical Imaging Group](https://bigwww.epfl.ch/). 
The distribution or the reproduction of the notebook is strictly prohibited without the written consent of the authors.  © EPFL 2026.

**Authors**: 
    [Pol del Aguila Pla](mailto:pol.delaguilapla@epfl.ch), 
    [Kay Lächler](mailto:kay.lachler@epfl.ch),
    [Alejandro Noguerón Arámburu](mailto:alejandro.nogueronaramburu@epfl.ch),
    [Yan Liu](mailto:yan.liu@epfl.ch),
    [Zhiyuan Hu](mailto:zhiyuan.hu@epfl.ch), and
    [Daniel Sage](mailto:daniel.sage@epfl.ch).
    
---

To ensure your work is graded correctly by our automated system, **do not create new cells or delete/rearrange/copy existing ones** when you submit. The current cells contain hidden metadata required for the auto-grader to identify your solutions. If you create temporary cells for testing during your work, remember to clean them up before submission.

# Lab 1.1: Pixel-wise operations
**Released**: Thursday, September 24, 2026

**Submission**: Friday, October 5, 2026 before 23:59 on [Moodle](https://moodle.epfl.ch/course/view.php?id=522)

**Grade weight**: Lab 1 (15 points), 9% of the overall grade

**Related lectures**: Chapters 1 and 2

Double-click on this cell, fill in your name and SCIPER number below, and run the next cell to verify your identity in Noto.

### Student Name:
### SCIPER:

In [ ]:
import getpass

# This line recovers your camipro number to mark the images with your ID
uid = (
    int(getpass.getuser().split("-")[2])
    if len(getpass.getuser().split("-")) > 2
    else ord(getpass.getuser()[0])
)
print(f"SCIPER: {uid}")

### Imports
In the next cell we import the Python libraries that we will use throughout the lab, as well as the `imviewer` class (a Python package developed specifically for these laboratories, see the documentation [here](https://github.com/Biomedical-Imaging-Group/interactive-kit/wiki/Image-Viewer), or run `help(viewer)` after loading the class):
* [`matplotlib.pyplot`](https://matplotlib.org/), to display images,
* [`ipywidgets`](https://ipywidgets.readthedocs.io/en/latest/), to make the image display interactive,
* [`numpy`](https://numpy.org/doc/stable/reference/), for mathematical operations on arrays, and
* [`scikit-image`](https://scikit-image.org/), to read the images and to assemble several slices into one montage.

Finally, we load the images used in this lab and make sure that they are of type `float64`, for higher accuracy during operations.

**Note:** Always run the previous cell and the next one before starting to work on the notebook.

In [ ]:
# Configure plotting as dynamic
%matplotlib widget

# Import required packages for this lab
import matplotlib.pyplot as plt
import ipywidgets as widgets
import numpy as np
from skimage import io
from skimage.util import montage
from interactive_kit import imviewer as viewer

# Load the images as float64
c_elegans = io.imread("images/c-elegans.tif").astype(np.float64)
pier = io.imread("images/pier.jpg").astype(np.float64)

# Pixel-wise operations (8 points)

In this laboratory you will learn the basics of pixel-wise image processing: operations that transform every pixel of an image independently of its neighbors. Despite their simplicity, pixel-wise operations have many practical applications, such as image normalization, contrast adjustment and artistic effects. They are also the natural place to get comfortable with the way images are handled in Python: as NumPy arrays, on which a pixel-wise operation is written as a single array expression instead of a loop over the pixels.

### Image visualization
Run the next cell and use the slider to cycle through the images to get familiar with them.

In [ ]:
plt.close("all")
pier_viewer = viewer(
    [pier] + list(c_elegans),
    title=["pier"] + [f"c_elegans {i + 1}" for i in range(c_elegans.shape[0])],
    hist=True,
    widgets=True,
)

# 1. Image normalization

Normalization is one of the key preprocessing steps in image processing. In this section, you will learn different ways to normalize an image, and see why it matters on a real microscopy dataset.

## 1.1. Pixel-wise operations in NumPy

Written naively, a pixel-wise operation is two nested `for` loops that visit every pixel one after the other. In pure Python this is slow: every iteration is interpreted separately. NumPy instead applies an operation to the whole array at once, in compiled code. Run the next cell to compare both ways of computing the mean of the image `pier`.

In [ ]:
import time


# The naive way: visit every pixel with two nested for-loops
def mean_with_loops(img):
    total = 0.0
    for y in range(img.shape[0]):
        for x in range(img.shape[1]):
            total += img[y, x]
    return total / (img.shape[0] * img.shape[1])


start = time.perf_counter()
mean_loops = mean_with_loops(pier)
time_loops = time.perf_counter() - start

# The NumPy way: one function call on the whole array
start = time.perf_counter()
mean_numpy = np.mean(pier)
time_numpy = time.perf_counter() - start

print(f"With for-loops: mean = {mean_loops:.4f}, computed in {1e3 * time_loops:.1f} ms")
print(
    f"With NumPy:     mean = {mean_numpy:.4f}, computed in {1e3 * time_numpy:.3f} ms ({time_loops / time_numpy:.0f} times faster)"
)

The result is the same, NumPy is orders of magnitude faster, and the code is much shorter. This is why, in these labs, **for-loops over the pixels of an image are not accepted**: every pixel-wise operation must be written as an array expression. For example, subtracting the mean from every pixel of a small $3 \times 3$ image is a single line. Run the next cell to see it.

In [ ]:
small_img = np.array([[0, 1, 2], [3, 4, 5], [6, 7, 8]], dtype=np.float64)

# A pixel-wise operation on the whole array at once: NumPy subtracts the scalar from every element
print(small_img - np.mean(small_img))

## 1.2. Normalization functions (3 points)

You will now implement three functions, which output images normalized with respect to different statistics:

* `make_zero_mean(img)`: normalizes the image so that the **sample mean of the pixel values is zero**. The standard deviation remains the same.
* `stretch_contrast(img)`: normalizes the image so that the **minimum value is $0$ and the maximum value is $1$** (the ratio $\frac{\sigma}{\mathrm{range}}$ does not change).
* `normalize_2nd_order_statistics(img)`: normalizes the image so that the **sample mean of the pixel values is zero** and the **sample standard deviation is $1$**.

*Hint:* NumPy provides [`np.mean`](https://numpy.org/doc/stable/reference/generated/numpy.mean.html), [`np.std`](https://numpy.org/doc/stable/reference/generated/numpy.std.html), [`np.min`](https://numpy.org/doc/stable/reference/generated/numpy.min.html) and [`np.max`](https://numpy.org/doc/stable/reference/generated/numpy.max.html). Each function can be written in a single line, and **for-loops are not accepted**. None of the functions should modify the input image `img`.

**For 1 point**, implement `make_zero_mean` in the cell below.

In [ ]:
# Function that normalizes the image so that the sample mean of the pixel values is zero.
def make_zero_mean(img):
    # Declare the output image
    output = np.zeros(img.shape)

    # YOUR CODE HERE

    # Return the output image
    return output

Run the next cell to test your implementation. We test on a simple $3 \times 3$ image and on the real image `pier`. If a warning is printed, your implementation is not yet correct.

In [ ]:
# Sanity check on a simple 3x3 image
test_img = np.array([[0, 1, 2], [3, 4, 5], [6, 7, 8]], dtype=np.float64)
test_zero_mean = make_zero_mean(test_img)
assert np.shape(test_zero_mean) == (3, 3), (
    "The output should have the same shape as the input."
)
if not np.allclose(test_zero_mean, [[-4, -3, -2], [-1, 0, 1], [2, 3, 4]]):
    print(
        "WARNING!\nThe function `make_zero_mean` is not yet correct: on the 3x3 test image, the output should be [[-4, -3, -2], [-1, 0, 1], [2, 3, 4]]."
    )
else:
    print("Nice, the function `make_zero_mean` subtracts the mean!")
if not np.array_equal(test_img, [[0, 1, 2], [3, 4, 5], [6, 7, 8]]):
    print(
        "WARNING!\nYour function modifies its input image. Compute the output as a new array instead of changing `img` in place."
    )

# Sanity check on a real image
if np.abs(np.mean(make_zero_mean(pier))) > 1e-8:
    print(
        "WARNING!\nThe mean of the output of `make_zero_mean` on the image `pier` is not 0."
    )
else:
    print("Nice, the function `make_zero_mean` works on real images!")

**For 1 point**, implement `stretch_contrast` in the cell below.

In [ ]:
# Function that normalizes the image so that all pixels have values between 0 and 1.
def stretch_contrast(img):
    # Declare the output image
    output = np.zeros(img.shape)

    # YOUR CODE HERE

    # Return the output image
    return output

Run the next cell to test your implementation on a simple $3 \times 3$ image and on the first slice of `c_elegans`.

In [ ]:
# Sanity check on a simple 3x3 image (whose minimum is not 0)
test_img = np.array([[2, 3, 4], [5, 6, 7], [8, 9, 10]], dtype=np.float64)
test_stretch = stretch_contrast(test_img)
assert np.shape(test_stretch) == (3, 3), (
    "The output should have the same shape as the input."
)
if not np.allclose(
    test_stretch, [[0, 0.125, 0.25], [0.375, 0.5, 0.625], [0.75, 0.875, 1]]
):
    print(
        "WARNING!\nThe function `stretch_contrast` is not yet correct: on the 3x3 test image, the output should be [[0, 0.125, 0.25], [0.375, 0.5, 0.625], [0.75, 0.875, 1]]. Remember to subtract the minimum before dividing by the range."
    )
else:
    print("Nice, the function `stretch_contrast` produces the correct output!")
if not np.array_equal(test_img, [[2, 3, 4], [5, 6, 7], [8, 9, 10]]):
    print(
        "WARNING!\nYour function modifies its input image. Compute the output as a new array instead of changing `img` in place."
    )

# Sanity check on a real image: the result must span exactly [0, 1]
test_stretch_contrast = stretch_contrast(c_elegans[0])
if np.min(test_stretch_contrast) != 0:
    print("WARNING!\nThe minimum value in the result of `stretch_contrast` is not 0.")
elif np.max(test_stretch_contrast) != 1:
    print("WARNING!\nThe maximum value in the result of `stretch_contrast` is not 1.")
else:
    print("Well done! Your function `stretch_contrast` works on real images.")

**For 1 point**, implement `normalize_2nd_order_statistics` in the cell below.

In [ ]:
# Function that normalizes the image so that the sample mean of the pixel values is 0 and the sample standard deviation is 1.
def normalize_2nd_order_statistics(img):
    # Declare the output image
    output = np.zeros(img.shape)

    # YOUR CODE HERE

    # Return the output image
    return output

Run the next cell to test your implementation. Both the biased and the unbiased estimator of the standard deviation are accepted.

In [ ]:
# Sanity check on a simple 3x3 image
test_img = np.array([[0, 1, 2], [3, 4, 5], [6, 7, 8]], dtype=np.float64)
test_normalize = normalize_2nd_order_statistics(test_img)
assert np.shape(test_normalize) == (3, 3), (
    "The output should have the same shape as the input."
)
expected_biased = (test_img - 4) / np.std(test_img)
expected_unbiased = (test_img - 4) / np.std(test_img, ddof=1)
if np.allclose(test_normalize, expected_biased):
    print(
        "Good job! The function `normalize_2nd_order_statistics` produces the correct output (biased estimator of the variance)."
    )
elif np.allclose(test_normalize, expected_unbiased):
    print(
        "Good job! The function `normalize_2nd_order_statistics` produces the correct output (unbiased estimator of the variance)."
    )
else:
    print(
        "WARNING!\nThe function `normalize_2nd_order_statistics` is not yet correct: subtract the mean, then divide by the standard deviation."
    )
if not np.array_equal(test_img, [[0, 1, 2], [3, 4, 5], [6, 7, 8]]):
    print(
        "WARNING!\nYour function modifies its input image. Compute the output as a new array instead of changing `img` in place."
    )

# Sanity check on a real image: mean 0 and standard deviation 1
test_normalize_statistics = normalize_2nd_order_statistics(c_elegans[0])
if np.abs(np.mean(test_normalize_statistics)) > 1e-8:
    print(
        "WARNING!\nThe mean of the result of `normalize_2nd_order_statistics` is not 0."
    )
elif not (
    np.isclose(np.std(test_normalize_statistics), 1, atol=1e-3)
    or np.isclose(np.std(test_normalize_statistics, ddof=1), 1, atol=1e-3)
):
    print(
        "WARNING!\nThe standard deviation of the result of `normalize_2nd_order_statistics` is not 1."
    )
else:
    print(
        "Well done! Your normalization of the 2nd order statistics works on real images."
    )

## 1.3. Application: photobleaching correction

In order for you to see the relevance of image normalization, we provide a sequence of fluorescence microscopy images (see more [here](https://en.wikipedia.org/wiki/Fluorescence_microscope)), named `c_elegans`. These are consecutive slices of the same 3D volume, but they appear darker over time due to photobleaching (the loss of fluorescence, read more [here](https://en.wikipedia.org/wiki/Photobleaching)). Of course, this is a substantial problem for the application, and to solve it we absolutely need to normalize these images.

The variable `c_elegans` is a 3D NumPy array of shape `(12, 92, 128)`: 12 slices of $92 \times 128$ pixels, so that `c_elegans[i]` is the $i$-th slice. Run the next cell to display the slices, together with a graph that shows the effect of photobleaching on the mean value of each slice.

**Note:** Make sure you explore the images and their histogram by moving the slider!

In [ ]:
# Show c_elegans images
plt.close("all")
viewer(
    [c_elegans[ind, :, :] for ind in range(12)],
    normalize=False,
    clip_range=[0, 255],
    title=[f"c_elegans {ind + 1}" for ind in range(12)],
    hist=True,
)
# Show decay of the mean value through time due to photobleaching
fig = plt.figure(num=f"SCIPER: {uid}", figsize=(6, 4))
plt.plot(
    [ind + 1 for ind in range(12)], [np.mean(c_elegans[ind, :, :]) for ind in range(12)]
)
plt.xticks([ind + 1 for ind in range(12)])
plt.xlabel("Image number")
plt.ylabel("Mean value")
plt.grid("both")
plt.show();

Now we are going to visualize the effect of each of your normalization functions on `c_elegans`. To compare all the slices at a glance, we assemble them into a single *montage* image with the function [`skimage.util.montage`](https://scikit-image.org/docs/stable/api/skimage.util.html#skimage.util.montage), which tiles the slices on a $4 \times 3$ grid. Run the next cell to declare the helper `make_montage` and to display the montage of the original slices next to the montages of the slices normalized with each of your three functions. Use the buttons `Next` and `Prev` to browse through the four images. If your implementations passed the previous tests, you should see the correct result. Look carefully at the images and answer the two multiple choice questions coming next.

In [ ]:
# Function that tiles the slices of a 3D stack on a 4 x 3 grid, optionally normalizing every slice first
def make_montage(stack, normalization=None):
    if normalization is not None:
        # Apply the normalization to each slice (a loop over the 12 slices, not over the pixels)
        stack = np.stack([normalization(img_slice) for img_slice in stack])
    return montage(stack, grid_shape=(4, 3))


# Define the lists of images and titles
image_list = [
    make_montage(c_elegans),
    make_montage(c_elegans, make_zero_mean),
    make_montage(c_elegans, stretch_contrast),
    make_montage(c_elegans, normalize_2nd_order_statistics),
]
title_list = [
    "Original c_elegans",
    "Zero Mean c_elegans",
    "Stretch Contrast c_elegans",
    "Normalize Statistics c_elegans",
]

# Display the montages
plt.close("all")
normalization_viewer = viewer(image_list, title=title_list, widgets=True)

### Multiple choice questions (1 point)

**For 0.5 points**, why does the bottom-right corner of the **Zero Mean** montage have lower contrast than the top-left corner?

1. Because the different subimages in the montage have different spreads of values around their mean.
2. It is an illumination effect.
3. The bottom-right subimages of the montage are defective.

Modify the variable `answer` in the next cell to reflect your choice.

In [ ]:
# Assign your answer to this variable
answer = None
# YOUR CODE HERE

In [ ]:
if not answer in [1, 2, 3]:
    print("WARNING!\nPossible answers are 1, 2 or 3.")

**For 0.5 points**, why is this not the case for the other two montages?

1. Because the images already had zero mean.
2. Because the other two functions modify the contrast by adjusting the range of intensities.

Modify the variable `answer` in the next cell to reflect your choice. As usual, there is another cell that will remind you to select a valid choice if you haven't.

In [ ]:
# assign your answer to this variable
answer = None
# YOUR CODE HERE

In [ ]:
if not answer in [1, 2]:
    print("WARNING!\nPossible answers are 1 and 2.")

### Normalizing a stack of images (1 point)

In `make_montage`, the normalization was applied slice by slice, with a loop over the 12 slices. Looping over slices is acceptable (it is not a loop over pixels), but NumPy can also normalize all the slices at once. The functions `np.mean` and `np.std` accept the argument `axis` to compute the statistics along the given axes only (here over the rows and the columns of each slice, i.e. `axis=(1, 2)`), and the argument `keepdims=True` to keep the result 3-dimensional, so that it *broadcasts* against the original stack.

**For 1 point**, implement the function `normalize_stack(stack)`, which takes a 3D array `stack` of shape `(n_slices, height, width)` and returns an array of the same shape in which **every slice has zero mean and unit standard deviation** (the normalization of `normalize_2nd_order_statistics`, applied to each slice separately). Write it as a single array expression using `axis=(1, 2)` and `keepdims=True` rather than a loop over the slices.

*Hint:* have a look at the documentation of [`np.mean`](https://numpy.org/doc/stable/reference/generated/numpy.mean.html) and read about [broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html).

In [ ]:
# Function that normalizes every slice of a 3D stack to zero mean and unit standard deviation
def normalize_stack(stack):
    # Declare the output
    output = np.zeros(stack.shape)

    # YOUR CODE HERE

    # Return the output
    return output

Run the next cell to test your implementation on a tiny stack of two $2 \times 3$ slices and on `c_elegans`.

In [ ]:
# Sanity check on a tiny stack of two 2x3 slices with different means and standard deviations
test_stack = np.array([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]], [[0.0, 2.0, 4.0], [6.0, 8.0, 10.0]]])
try:
    test_normalized = normalize_stack(test_stack)
except ValueError as err:
    raise AssertionError(
        f"`normalize_stack` failed on a stack of shape (2, 2, 3): {err}. Keep the per-slice statistics 3-dimensional (`keepdims=True`) so that they broadcast against the stack."
    ) from None
assert np.shape(test_normalized) == (2, 2, 3), (
    "The output should have the same shape as the input stack."
)
expected_biased = (test_stack - [[[2.5]], [[5.0]]]) / np.std(
    test_stack, axis=(1, 2), keepdims=True
)
expected_unbiased = (test_stack - [[[2.5]], [[5.0]]]) / np.std(
    test_stack, axis=(1, 2), keepdims=True, ddof=1
)
if np.allclose(test_normalized, expected_biased) or np.allclose(
    test_normalized, expected_unbiased
):
    print("Nice, every slice of the tiny stack is normalized!")
else:
    print(
        "WARNING!\nThe function `normalize_stack` is not yet correct: each slice should be normalized with its own mean and standard deviation."
    )
if not np.array_equal(
    test_stack, [[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]], [[0.0, 2.0, 4.0], [6.0, 8.0, 10.0]]]
):
    print(
        "WARNING!\nYour function modifies its input stack. Compute the output as a new array instead of changing `stack` in place."
    )

# Sanity check on c_elegans: every slice must have mean 0 and standard deviation 1
c_elegans_normalized = normalize_stack(c_elegans)
slice_means = np.mean(c_elegans_normalized, axis=(1, 2))
slice_stds = np.std(c_elegans_normalized, axis=(1, 2))
if np.shape(c_elegans_normalized) != c_elegans.shape:
    print("WARNING!\nThe output should have the same shape as `c_elegans`.")
elif np.max(np.abs(slice_means)) > 1e-8:
    print(
        f"WARNING!\nNot every slice has zero mean: slice {np.argmax(np.abs(slice_means))} has mean {slice_means[np.argmax(np.abs(slice_means))]:.4f}. Did you compute the statistics of each slice separately?"
    )
elif np.max(np.abs(slice_stds - 1)) > 1e-3:
    print(
        f"WARNING!\nNot every slice has unit standard deviation: slice {np.argmax(np.abs(slice_stds - 1))} has standard deviation {slice_stds[np.argmax(np.abs(slice_stds - 1))]:.4f}."
    )
else:
    print("Well done! Every slice of `c_elegans` is normalized.")

# 2. Solarization

[Solarization](https://en.wikipedia.org/wiki/Solarization_%28photography%29) is one of the oldest effects known in photography. It refers to the *tone reversal* effect that is caused by over-exposure of certain photographic papers during the development process. In this context, *tone reversal* means that the brightest objects (e.g. the sun) appear darker than the rest of the image. After its discovery, the solarization effect was used for artistic purposes. Of course, when digital photography came in, it stopped being an over-exposure effect, but it can still be artificially provoked for artistic purposes.

Solarization can be expressed mathematically (assuming that our photo has a range of $[0, 255]$) as
$$h[k, l] = \begin{cases}
    g[k, l],              & \text{if } g[k, l] < T\\
    T \cdot \frac{255 - g[k, l]}{255 - T},& \text{if } g[k, l] \geq T,
\end{cases}$$
where $g$ is the input image, $h$ is the solarized output image, and $T$ is the threshold defining the strength of the solarization: the pixels darker than $T$ are left unchanged, while the pixels brighter than $T$ are reversed in tone ($g = 255$ becomes $h = 0$), with a mapping that is continuous at $T$.

## 2.1. Implementation (2 points)

**For 2 points**, complete the function `solarize(img, T)` in the next cell, which implements the formula above and returns the solarized image, without modifying the input image `img`.

*Hint:* a condition on the pixel values, such as $g[k, l] \geq T$, is expressed in NumPy with a boolean mask (`img >= T`), which can be used to index the array ([boolean indexing](https://numpy.org/doc/stable/user/basics.indexing.html#boolean-array-indexing), as in the introductory lab) or as the condition of [`np.where`](https://numpy.org/doc/stable/reference/generated/numpy.where.html). Start from a copy of the input (`np.copy`) if you modify values in place. As always, for-loops are not accepted.

In [ ]:
# Function that applies the solarization effect to an image
def solarize(img, T):
    # Declare the output image
    output = np.copy(img)

    # YOUR CODE HERE

    # Return the output image
    return output

Run the next cell to test your implementation on a small image with $T = 100$.

In [ ]:
# Sanity check on a small image with T = 100
test_img = np.array([[0.0, 50.0, 100.0], [150.0, 200.0, 255.0]])
test_solarized = solarize(test_img, 100)
expected = np.array([[0.0, 50.0, 100.0], [100 * 105 / 155, 100 * 55 / 155, 0.0]])
assert np.shape(test_solarized) == (2, 3), (
    "The output should have the same shape as the input."
)
if not np.allclose(test_solarized, expected):
    print(
        f"WARNING!\nThe function `solarize` is not yet correct: with T = 100, the input {test_img.tolist()} should give {np.round(expected, 2).tolist()}, but you got {np.round(test_solarized, 2).tolist()}."
    )
else:
    print("Nice, the function `solarize` implements the formula!")
if not np.array_equal(test_img, [[0.0, 50.0, 100.0], [150.0, 200.0, 255.0]]):
    print(
        "WARNING!\nYour function modifies its input image. Start from a copy of the input (`np.copy`)."
    )

## 2.2. Interactive exploration

Now let's apply your function to the image `pier` with a threshold of $150$ and compare the result to the original image. Run the next cell to display both. If you are unsure whether they differ, you can go to the `Options` menu of the viewer and click `Compare`: the differences between the two images are then highlighted in red.

In [ ]:
plt.close("all")
view = viewer(
    [pier, solarize(pier, 150)],
    title=["Original pier", "Solarized pier (T = 150)"],
    subplots=[1, 2],
    widgets=True,
)

In the next cell, we provide you with an interactive viewer, where you can play around with the threshold value $T$ to see the different effects this can have on the image. To do so, go to the menu `Extra Widgets`, change the threshold `T`, and click on the `Solarize` button. If you want to see the original image again, just click `Reset`.

In [ ]:
# Declare the extra widgets
T_slider = widgets.IntSlider(value=150, min=0, max=254, step=1, description="T")
button = widgets.Button(description="Solarize")


# Callback that applies your function with the current value of the slider
def solarize_callback(img):
    return solarize(img, T_slider.value)


plt.close("all")
view = viewer(
    pier,
    title=["Solarized pier"],
    new_widgets=[T_slider, button],
    callbacks=[solarize_callback],
    widgets=True,
)

## 2.3. Properties of the solarization operator (1 point)

**For 0.5 points each**, answer the following two questions, assuming a fixed parameter $T$. Think of the function `solarize` as an operator.

Q1. Is `solarize` a linear operator?

Q2. Is `solarize` a shift-invariant operator? (That is, if the input image is shifted by a fixed spatial offset, is the output shifted by the same offset?)

In the following cell, change the values of `q1_ans` and `q2_ans` to reflect your answers. **Assign a boolean (`True` or `False`)** to each variable. Then, run the following two cells to check if your choices are valid.

In [ ]:
# Change the values of the following variables
q1_ans = None
q2_ans = None
# YOUR CODE HERE

In [ ]:
if not isinstance(q1_ans, bool):
    print("WARNING!\nq1_ans should be a boolean variable (True or False)")

In [ ]:
if not isinstance(q2_ans, bool):
    print("WARNING!\nq2_ans should be a boolean variable (True or False)")

🎉 Congratulations on finishing the first part of the Pixel-Fourier lab!

Make sure to save your notebook (you might want to keep a copy on your personal computer) and upload it to Moodle, **in a zip file with the other notebook of this lab.**

* Keep the name of the notebook as: *1_pixelwise_operations.ipynb*,
* Name the zip file: *pixel_fourier_lab.zip*.